In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import shutil
import yaml

# Change only this if your folder is in another location in Google Drive.
DRIVE_DATASET_DIR = Path("/content/drive/MyDrive/Classroom_v2")
LOCAL_DATASET_DIR = Path("/content/Classroom_v2")

if not DRIVE_DATASET_DIR.exists():
    raise FileNotFoundError(f"Not found: {DRIVE_DATASET_DIR}")

if LOCAL_DATASET_DIR.exists():
    shutil.rmtree(LOCAL_DATASET_DIR)

shutil.copytree(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)

classes = [
    "bowed_down",
    "non_attentive",
    "sleeping",
    "standing",
    "watching_phone",
    "working",
]

for split in ["train", "valid", "test"]:
    images = list((LOCAL_DATASET_DIR / split / "images").glob("*.*"))
    labels = list((LOCAL_DATASET_DIR / split / "labels").glob("*.txt"))
    print(f"{split}: {len(images)} images, {len(labels)} labels")

DATA_YAML = Path("/content/data.yaml")

config = {
    "path": str(LOCAL_DATASET_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(classes),
    "names": classes,
}

with open(DATA_YAML, "w") as file:
    yaml.dump(config, file, sort_keys=False)

print("\nCreated:")
print(DATA_YAML.read_text())

In [ ]:
!pip install -q -U ultralytics

import os
os.environ["WANDB_MODE"] = "disabled"

from ultralytics import YOLO
import ultralytics

print("Ultralytics version:", ultralytics.__version__)

In [ ]:
from ultralytics import YOLO

# Load pretrained weights (Recommended)
model = YOLO("yolo12m.pt")

results = model.train(
    data="/content/data.yaml",
    epochs=100,
    patience=20,
    imgsz=1280,               # High resolution for back-row detection
    batch=4,                  # Optimal batch size for 1280px on T4 GPU
    device=0,
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,
    warmup_epochs=3,
    mosaic=1.0,               # Great for small object detection
    mixup=0.0,
    amp=True,
    workers=2,
    project="/content/runs/yolo12m_finetune",
    name="classroom_v1",
    seed=42,
    plots=True,
)

BEST_MODEL = "/content/runs/yolo12m_finetune/classroom_v1/weights/best.pt"

print("Training complete.")
print("Best model path:", BEST_MODEL)


In [ ]:
from ultralytics import YOLO

# Load the fine-tuned best model weights
trained_model = YOLO(BEST_MODEL)

# Run validation on the validation set
metrics = trained_model.val(
    data="/content/data.yaml",
    split="val",
    imgsz=1280,   # Match the training image size for accurate evaluation
    device=0,
    plots=True,
)

# Output evaluation metrics
print(f"Precision (mp): {metrics.box.mp:.4f}")
print(f"Recall (mr): {metrics.box.mr:.4f}")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")


In [ ]:
from pathlib import Path
import shutil

SAVE_FOLDER = Path("/content/drive/MyDrive/YOLOv12_Models")
SAVE_FOLDER.mkdir(exist_ok=True)

saved_model = SAVE_FOLDER / "classroom_yolo12m_v2.pt"
shutil.copy2(BEST_MODEL, saved_model)

print("Saved:", saved_model)

In [ ]:
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Video, display

BEST_MODEL = "/content/drive/MyDrive/YOLOv12_Models/classroom_yolo12m_v2.pt"
VIDEO_PATH = "/content/44 secs clip.mp4"

model = YOLO(BEST_MODEL)

OUTPUT_DIR = Path("/content/video_predictions")
OUTPUT_NAME = "classroom_yolo12m_tracked"

# Swapped model.predict for model.track and added ByteTrack configuration
tracking_stream = model.track(
    source=VIDEO_PATH,
    conf=0.42,
    imgsz=1280,
    tracker="bytetrack.yaml",  # Uses ByteTrack to maintain temporal consistency
    stream=True,
    save=True,
    verbose=False,
    project=str(OUTPUT_DIR),
    name=OUTPUT_NAME,
    exist_ok=True,
    vid_stride=1,
)

# Iterate through generator to process the full video
for _ in tracking_stream:
    pass

result_folder = OUTPUT_DIR / OUTPUT_NAME

result_video = next(
    file for file in result_folder.iterdir()
    if file.suffix.lower() in {".mp4", ".avi", ".mov"}
)

print("Annotated tracked video:", result_video)
display(Video(str(result_video), embed=True, width=900))


In [ ]:
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Video, display
import subprocess
import shutil

# Do not mount Drive again.
MODEL_PATH = Path(
    "/content/drive/MyDrive/YOLOv12_Models/classroom_yolo12m_v2.pt"
)

# Your uploaded video from the error message.
ORIGINAL_VIDEO = Path("/content/1792595038871265.mp4")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model missing: {MODEL_PATH}")

if not ORIGINAL_VIDEO.exists():
    raise FileNotFoundError(f"Video missing: {ORIGINAL_VIDEO}")

# Convert input to a format OpenCV and Ultralytics reliably support.
SAFE_VIDEO = Path("/content/classroom_input_h264.mp4")

subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", str(ORIGINAL_VIDEO),
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-c:a", "aac",
        "-movflags", "+faststart",
        str(SAFE_VIDEO),
    ],
    check=True,
)

print("Converted input video:", SAFE_VIDEO)

model = YOLO(str(MODEL_PATH))

OUTPUT_ROOT = Path("/content/video_predictions")
RUN_NAME = "classroom_yolo12m_tracked_fixed"
RUN_DIR = OUTPUT_ROOT / RUN_NAME

if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)

# stream=True prevents memory crashes on long clips.
for _ in model.track(
    source=str(SAFE_VIDEO),
    conf=0.35,
    imgsz=1280,
    tracker="bytetrack.yaml",
    persist=True,
    stream=True,
    save=True,
    verbose=False,
    project=str(OUTPUT_ROOT),
    name=RUN_NAME,
    exist_ok=True,
    vid_stride=1,
):
    pass

result_video = next(
    file for file in RUN_DIR.iterdir()
    if file.suffix.lower() in {".mp4", ".avi", ".mov"}
)

# Convert output AVI if needed so Colab can play it.
PLAYABLE_VIDEO = result_video.with_suffix(".mp4")

if result_video.suffix.lower() != ".mp4":
    subprocess.run(
        [
            "ffmpeg", "-y",
            "-i", str(result_video),
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(PLAYABLE_VIDEO),
        ],
        check=True,
    )
else:
    PLAYABLE_VIDEO = result_video

print("Annotated video:", PLAYABLE_VIDEO)
display(Video(str(PLAYABLE_VIDEO), embed=True, width=900))